# Counterfactual explanations for a trained pipeline

A counterfactual answers this question: *what must change in this input, so that the prediction moves into a target range?*

The `CounterfactualEvaluator` uses the [CELIA](https://github.com/serval-uni-lu/celia) library. It:

1. Rebuilds the trained pipeline and inserts a `PerturbationNode`, by default right after the data source.
2. Wraps the part of the pipeline after the `PerturbationNode` as a black-box model for CELIA.
3. Runs each scenario (a CELIA method with its settings) on each sample, in parallel with Ray.
4. Scores each counterfactual: validity, distance to the original row, number of changed columns, and constraint violations.

The original pipeline and its trained parameters do not change.

CELIA is an optional dependency:

```bash
uv sync --extra counterfactuals        # or: pip install "nodeml[counterfactuals]"
```

In [ ]:
import numpy as np
import pandas as pd

from nodeml import NODE_REGISTRY
from nodeml.components.nodes.data_sources.inputs_passthrough import (
    InputsPassthroughConfig,
)
from nodeml.core.common.data.data import (
    ArrayLikeEnum,
    CategoricalData,
    DataCategoryEnum,
    DataStructureEnum,
    NumericalData,
    TabularDataContext,
)
from nodeml.core.common.enums import NodeExecutionMode
from nodeml.core.nodes.node import Port
from nodeml.core.pipeline.pipeline import Edge, Pipeline
from nodeml.core.pipeline.runners.smart_runner import SmartRunner, SmartRunnerConfig

## 1. Train a pipeline

The data is synthetic. The target is the yearly energy cost of a house. It depends on the surface, the insulation, the age, and the heating type (a categorical column).

The pipeline splits the numerical and the categorical columns, scales the first, one-hot encodes the second, and trains a random forest.

In [ ]:
rng = np.random.default_rng(0)
n = 400
X = pd.DataFrame(
    {
        "surface_m2": rng.uniform(40, 200, n).round(1),
        "insulation_cm": rng.uniform(0, 30, n).round(1),
        "age_years": rng.integers(0, 80, n).astype(np.int64),
        "heating": rng.choice(["gas", "oil", "heat_pump"], n),
    }
)
factor = X["heating"].map({"gas": 1.0, "oil": 1.3, "heat_pump": 0.5})
y = pd.DataFrame(
    {
        "cost_eur": (
            X["surface_m2"] * factor * (12 - 0.3 * X["insulation_cm"])
            + 4 * X["age_years"]
        ).round(1)
    }
)
X_ctx = TabularDataContext(
    list(X.columns),
    list(X.dtypes),
    [NumericalData, NumericalData, NumericalData, CategoricalData],
)
y_ctx = TabularDataContext(["cost_eur"], list(y.dtypes), [NumericalData])
X.head()

In [ ]:
source_config = InputsPassthroughConfig(
    out_ports={
        "X": Port(
            arr_type=ArrayLikeEnum.PANDAS,
            data_structure=DataStructureEnum.TABULAR,
            data_category=DataCategoryEnum.MIXED,
            data_shape="batch feature",
            desc="House features.",
        ),
        "y": Port(
            arr_type=ArrayLikeEnum.PANDAS,
            data_structure=DataStructureEnum.TABULAR,
            data_category=DataCategoryEnum.NUMERICAL,
            data_shape="batch 1",
            desc="Yearly energy cost.",
            mode=[NodeExecutionMode.TRAINING, NodeExecutionMode.EVALUATION],
        ),
    }
)

model_config = NODE_REGISTRY.get_node_config_class("RandomForestRegressor")()
model_config.running_config.random_state = 0

pipe = Pipeline()
pipe.add_node("source", "InputsPassthrough", source_config)
pipe.add_node("split", "DataCategoryFilter")
pipe.add_node("scaler", "StandardScaler")
pipe.add_node("onehot", "OneHotEncoding")
pipe.add_node("concat", "FeatureConcatenate")
pipe.add_node("model", "RandomForestRegressor", model_config)
pipe.add_node("sink", "Sink")
pipe.add_edge(Edge(source="source", target="split", ports_map=[("X", "input")]))
pipe.add_edge(Edge(source="split", target="scaler", ports_map=[("numerical", "input")]))
pipe.add_edge(
    Edge(source="split", target="onehot", ports_map=[("categorical", "input")])
)
pipe.add_edge(Edge(source="scaler", target="concat", ports_map=[("output", "input_1")]))
pipe.add_edge(Edge(source="onehot", target="concat", ports_map=[("output", "input_2")]))
pipe.add_edge(Edge(source="concat", target="model", ports_map=[("output", "X")]))
pipe.add_edge(Edge(source="source", target="model", ports_map=[("y", "y")]))
pipe.add_edge(Edge(source="model", target="sink", ports_map=[("pred", "pred")]))
pipe.compile()

runner = SmartRunner(pipe, config=SmartRunnerConfig(verbose=False))
runner.train(input_data={"source": {"X": (X, X_ctx), "y": (y, y_ctx)}})

## 2. Create the evaluator

`CounterfactualEvaluator.from_pipeline` copies the config and the trained parameters of the pipeline. You can also use `CounterfactualEvaluator.from_dir(...)` for a pipeline saved with `save_config_to_dir` and `save_params_to_dir`.

The default insertion point is the source port that the model needs in inference: here `source.X`. The counterfactuals are therefore rows of the raw data, with the original column names. To insert the `PerturbationNode` later, for example after preprocessing, set `InsertionPoint(node=..., port=...)`.

The constraints refer to the columns at the insertion point:

* `immutable_columns`: the counterfactuals must not change them. Here, a house cannot become younger.
* `feasible_ranges`: the allowed `(min, max)` range of a numerical column.
* `feasible_categories`: the allowed values of a categorical column.
* `frozen_columns`: CELIA does not see them at all. Use this for identifiers and for columns with many categories.

In [ ]:
from nodeml.core.pipeline.counterfactuals import (
    CounterfactualEvaluator,
    CounterfactualEvaluatorConfig,
    CounterfactualReport,
    CounterfactualScenario,
    FeatureConstraints,
    TargetRange,
)

evaluator = CounterfactualEvaluator.from_pipeline(
    pipe,
    config=CounterfactualEvaluatorConfig(
        use_ray=True,  # one Ray task for each (scenario, sample) pair
        constraints=FeatureConstraints(
            immutable_columns=["age_years"],
            feasible_ranges={"insulation_cm": (0.0, 40.0), "surface_m2": (40.0, 200.0)},
        ),
    ),
)
evaluator.points

## 3. Define the scenarios

A scenario is one CELIA method with its settings and a target range. `TargetRange` has three kinds:

* `"absolute"`: the prediction must be in `[low, high]`;
* `"delta"`: in `[p + low, p + high]`, where `p` is the baseline prediction of the sample;
* `"relative"`: in `[p * (1 + low), p * (1 + high)]`.

Here we ask for a cost 20 % to 40 % lower.

In [ ]:
target = TargetRange(kind="relative", low=-0.4, high=-0.2)
scenarios = [
    CounterfactualScenario(
        name="dice_random",
        method="dice",
        target=target,
        generate_kwargs={"total_CFs": 3, "random_seed": 0},
    ),
    CounterfactualScenario(
        name="dice_genetic",
        method="dice",
        target=target,
        explainer_kwargs={"method": "genetic"},
        generate_kwargs={"total_CFs": 3},
    ),
    CounterfactualScenario(
        name="nnce",
        method="nnce",
        target=target,
        generate_kwargs={"n_counterfactuals": 3},
    ),
]

## 4. Evaluate

`samples` are the rows to explain, at the insertion point. `reference_input` is the runner input for the reference rows: CELIA uses them as the data distribution. With a data source that loads its own data (for example a `TabularCSVFetcher`), leave it at `None`.

In [ ]:
report = evaluator.evaluate(
    scenarios,
    samples=X.iloc[:5],
    reference_input={"source": {"X": (X, X_ctx)}},
)
report.summary_frame()

In [ ]:
report.samples_frame()

`changes_frame` shows the original row and its counterfactuals. `"-"` marks a value that did not change.

In [ ]:
report.changes_frame("dice_random", "0")

In [ ]:
# NNCE returns real reference rows, so it does not find a counterfactual for every sample.
found = report.samples_frame().query("scenario == 'nnce' and status == 'found'")
report.changes_frame("nnce", found["sample_id"].iloc[0])

Every metric of every counterfactual is in `counterfactuals_frame()`.

The evaluator checks the constraints itself. A method can break them: in this run, DiCE `"genetic"` sets the immutable `age_years` to 0 for one sample. The `violations` column shows this.

In [ ]:
report.counterfactuals_frame()[
    [
        "scenario",
        "sample_id",
        "prediction",
        "valid",
        "l1_distance",
        "n_changed",
        "violations",
    ]
].head(10)

## 5. Save the report

The report is a pydantic model. It saves to JSON and loads back.

In [ ]:
import tempfile
from pathlib import Path

path = report.save_json(Path(tempfile.mkdtemp()) / "counterfactual_report.json")
CounterfactualReport.load_json(path) == report

## Notes

* **Methods.** The evaluator supports the CELIA regression methods that work with a black-box model: DiCE (`"random"`, `"genetic"` and `"kdtree"`) and NNCE. BugDoc is not supported yet: the CELIA 0.1.0 BugDoc regressor always fails.
* **Missing values.** CELIA does not accept them. With `missing_values="freeze"` (the default), a column that is missing in the sample is frozen, and the missing reference values are filled for CELIA only. The pipeline always gets the real rows.
* **Categorical columns.** DiCE `"random"` and NNCE get integer codes instead of string categories. NNCE then measures distances between codes, which is only an approximation.
* **Validity.** The evaluator predicts every counterfactual again with the pipeline. A counterfactual is valid only if this prediction is in the target range.